# Capítulo 7 — Entender el cálculo de una red neuronal · Solución comentada

*Matemáticas para la IA, paso a paso*, Belk Siri — notebook complementario.

[Abrir en Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/es/ch07_corrige.ipynb) · Página complementaria: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/es.html

Cada respuesta empieza con el razonamiento en papel y después muestra su equivalente en Python.

**Índice.** Datos · Los cálculos de las lecciones 7.1 a 7.3 · Lección 7.4 (minilotes y épocas) · Taller opcional (doce mensajes) · Ejercicios 7.1 a 7.6

**Antes de abrir Jupyter:** haz primero a mano la propagación hacia delante de M7 (lección 7.1) y la derivada respecto a $w_{11}$ (lección 7.2). Las celdas de abajo sirven para **comprobar** esos cálculos.

**Código de colores:** índigo = cálculo hacia delante; magenta discontinuo = camino del gradiente; trazo grueso ámbar = peso estudiado $w_{11}$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Código de colores del libro: cálculo hacia delante en índigo, gradiente en magenta (discontinuo), peso estudiado en ámbar grueso
FORWARD, BACKWARD, STUDIED_WEIGHT = '#3949ab', '#c2185b', '#f9a825'


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def loss(y, p):
    # pérdida logarítmica de un mensaje (ecuación 5.6)
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)))


def forward(P, x):
    # Propagación hacia delante (ecuaciones 7.1 y 7.2): devuelve un diccionario z1, a1, z2, a2, z, p
    z1 = P['w11'] * x[0] + P['w12'] * x[1] + P['b1']
    z2 = P['w21'] * x[0] + P['w22'] * x[1] + P['b2']
    a1, a2 = sigmoid(z1), sigmoid(z2)
    z = P['v1'] * a1 + P['v2'] * a2 + P['c']
    return {'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z': z, 'p': sigmoid(z)}


def draw(P, x, backward=False):
    # El esquema del libro, simplificado: índigo para la ida, magenta para la vuelta, w11 en ámbar grueso
    v = forward(P, x)
    pos = {'x1': (0, 3), 'x2': (0, 0), 'n1': (3, 3), 'n2': (3, 0), 's': (6, 1.5)}
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for a, b, name in (('x1', 'n1', 'w11'), ('x2', 'n1', 'w12'), ('x1', 'n2', 'w21'), ('x2', 'n2', 'w22'),
                      ('n1', 's', 'v1'), ('n2', 's', 'v2')):
        (xa, ya), (xb, yb) = pos[a], pos[b]
        if name == 'w11':
            ax.plot([xa, xb], [ya, yb], color=STUDIED_WEIGHT, linewidth=7, zorder=1)
        ax.annotate('', xy=(xb - 0.35, yb + (ya - yb) * 0.12), xytext=(xa + 0.3, ya),
                    arrowprops=dict(arrowstyle='-|>', color=FORWARD), zorder=2)
        t = 0.3 if a[0] == 'x' and b != ('n1' if a == 'x1' else 'n2') else 0.5
        ax.text(xa + t * (xb - xa), ya + t * (yb - ya) + 0.15, f"{name} = {P[name]:g}" + (' (peso estudiado)' if name == 'w11' else ''),
                color=FORWARD, fontsize=8, ha='center', bbox=dict(facecolor='white', edgecolor='none', alpha=0.85))
    texts = {'x1': f"x1 = {x[0]}", 'x2': f"x2 = {x[1]}",
              'n1': f"N1\nz1 = {v['z1']:.3f}\na1 = {v['a1']:.4f}", 'n2': f"N2\nz2 = {v['z2']:.3f}\na2 = {v['a2']:.4f}",
              's': f"salida\nz = {v['z']:.3f}\np̂ = {v['p']:.3f}"}
    for k, (cx, cy) in pos.items():
        ax.text(cx, cy, texts[k], ha='center', va='center', fontsize=8,
                bbox=dict(boxstyle='circle', facecolor='white', edgecolor=FORWARD if k[0] != 'x' else 'grey'))
    if backward:
        for (x0, y0), (x1, y1), lab in (((6.6, 2.4), (3.5, 3.6), '② v1'), ((3.0, 3.9), (1.4, 3.4), '③ a1(1 − a1), ④ x1')):
            ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                        arrowprops=dict(arrowstyle='-|>', color=BACKWARD, linestyle='--', connectionstyle='arc3,rad=0.3'))
            ax.text((x0 + x1) / 2, max(y0, y1) + 0.35, lab, color=BACKWARD, fontsize=8, ha='center')
        ax.text(6.6, 2.7, '① p̂ − y', color=BACKWARD, fontsize=8)
    ax.set_xlim(-0.8, 7.4); ax.set_ylim(-0.8, 4.4); ax.axis('off'); plt.show()

## Datos

Los identificadores D7.1, D7.E1 y D5.1 son los del libro y de los archivos CSV.

In [ ]:
# Datos del capítulo 7, copiados de los archivos CSV del libro (no modificar)

# D7.1 — los nueve parámetros de la red de las lecciones 7.1 a 7.3
network = {'w11': 0.5, 'w12': 0.5, 'b1': -2, 'w21': 1, 'w22': -1, 'b2': 0, 'v1': 2, 'v2': -2, 'c': 0}

# D7.E1 — la red del colega (ejercicios 7.1 y 7.2)
colleague = {'w11': 1, 'w12': 1, 'b1': -4, 'w21': -1, 'w22': 2, 'b2': -1, 'v1': 2, 'v2': 1, 'c': -2}

# D5.1 — los doce mensajes del capítulo 5: etiqueta, enlaces, palabras sospechosas
twelve = pd.DataFrame([
    ('M1', 'spam', 5, 4),
    ('M2', 'legitime', 2, 0),
    ('M3', 'legitime', 4, 0),
    ('M4', 'spam', 3, 3),
    ('M5', 'legitime', 1, 0),
    ('M6', 'legitime', 2, 2),
    ('M7', 'spam', 2, 3),
    ('M8', 'legitime', 2, 1),
    ('M9', 'spam', 4, 4),
    ('M10', 'legitime', 1, 1),
    ('M11', 'legitime', 0, 0),
    ('M12', 'legitime', 0, 1)
], columns=['message', 'label', 'links', 'suspicious_words'])

M7 = (2, 3)   # el mensaje M7: 2 enlaces, 3 palabras sospechosas, spam (y = 1)
M3 = (4, 0)   # el boletín M3: 4 enlaces, 0 palabras sospechosas, legítimo (y = 0)

### Las variables de los datos

Los nombres del código están en inglés. Para las columnas, la última columna da el nombre utilizado en los archivos CSV del libro.

| Nombre | Significado | Unidad | Columna del CSV |
|:--|:--|:--|:--|
| `message` | identificador del mensaje (M1…) | — | — |
| `label` | etiqueta real: spam o legítimo | — | `etiquette` |
| `links` | número de enlaces en el mensaje | enlaces | `liens` |
| `suspicious_words` | número de palabras sospechosas en el mensaje | palabras | `mots_suspects` |
| `network` | los nueve parámetros de la red de la lección 7.1 (pesos, sesgos, constante) | — | — |
| `colleague` | los nueve parámetros de la red del colega | — | — |
| `twelve` | los doce mensajes del capítulo 5 | — | — |

## Los cálculos de las lecciones, en Python

Estas celdas rehacen los cálculos de las lecciones 7.1 a 7.3.

In [ ]:
# Lección 7.1 — propagación hacia delante del mensaje M7 (ecuaciones 7.1 y 7.2)
v = forward(network, M7)
print(f"N1: z1 = {v['z1']:.3f}, a1 = {v['a1']:.4f}")
print(f"N2: z2 = {v['z2']:.3f}, a2 = {v['a2']:.4f}")
print(f"salida: z = {v['z']:.3f}, p̂ = {v['p']:.3f}, pérdida = {loss(1, v['p']):.3f}")
print(f"M3 (boletín): a2 = {forward(network, M3)['a2']:.3f}, p̂ = {forward(network, M3)['p']:.3f}")
draw(network, M7)

In [ ]:
# Lección 7.2 — la regla de la cadena para w11 (ecuación 7.3), factor por factor
f1 = v['p'] - 1                  # ① pérdida ← z: p̂ − y
f2 = network['v1']                # ② z ← a1: v1
f3 = v['a1'] * (1 - v['a1'])     # ③ a1 ← z1: derivada de la sigmoide, con la ACTIVACIÓN a1
f4 = M7[0]                       # ④ z1 ← w11: x1
d_w11 = f1 * f2 * f3 * f4
print(f"① {f1:.3f}  ② {f2:g}  ③ {f3:.4f}  ④ {f4}  → derivada = {d_w11:.3f}")

# Comprobación con una pequeña perturbación numérica (como las secantes de la lección 3.3)
def loss_w11(w):
    return loss(1, forward(dict(network, w11=w), M7)['p'])
h = 0.0001
print(f"derivada numérica: ({loss_w11(0.5 + h):.6f} − {loss_w11(0.5 - h):.6f}) / (2h) = "
      f"{(loss_w11(0.5 + h) - loss_w11(0.5 - h)) / (2 * h):.4f}")

# Una actualización a mano, η = 0,5, y luego rehacemos la propagación hacia delante
eta = 0.5
w11_new = round(0.5 - eta * d_w11, 3)   # redondeado a la milésima, como en el libro: 0,655
v_n = forward(dict(network, w11=w11_new), M7)
print(f"w11: 0,5 → {w11_new:.3f}; z1 = {v_n['z1']:.2f}, a1 = {v_n['a1']:.4f}, z = {v_n['z']:.3f}, "
      f"p̂ = {v_n['p']:.3f}, pérdida = {loss(1, v_n['p']):.3f}")
draw(network, M7, backward=True)

ws = np.linspace(-1, 3, 200)
plt.figure(figsize=(6, 3))
plt.plot(ws, [loss_w11(w) for w in ws], color='dimgrey', label='pérdida de M7 según w11')
plt.plot(ws[25:126], loss_w11(0.5) + d_w11 * (ws[25:126] - 0.5), '--', color=BACKWARD, label='tangente en 0,5')
plt.scatter([0.5, w11_new], [loss_w11(0.5), loss_w11(w11_new)], color=['black', FORWARD], zorder=3)
plt.xlabel('w11'); plt.ylabel('pérdida de M7'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Lección 7.3 — retropropagación: las nueve derivadas (ecuación 7.4) y luego una actualización de TODOS los parámetros
def backward(P, x, y):
    v = forward(P, x)
    d = v['p'] - y                                   # señal de error de la salida
    d1 = d * P['v1'] * v['a1'] * (1 - v['a1'])       # señal de error de N1
    d2 = d * P['v2'] * v['a2'] * (1 - v['a2'])       # señal de error de N2
    return {'w11': d1 * x[0], 'w12': d1 * x[1], 'b1': d1, 'w21': d2 * x[0], 'w22': d2 * x[1], 'b2': d2,
            'v1': d * v['a1'], 'v2': d * v['a2'], 'c': d}


def update(P, g, eta):
    return {k: P[k] - eta * g[k] for k in P}        # todas las derivadas se toman ANTES de modificar


g = backward(network, M7, 1)
new = update(network, g, 0.5)
print(pd.DataFrame({'valor': network, 'derivada': g, 'nuevo valor': new}).round(3))
print(f"M7: p̂ = {forward(new, M7)['p']:.3f}, pérdida = {loss(1, forward(new, M7)['p']):.3f}")
print(f"M3 antes: p̂ = {forward(network, M3)['p']:.3f}, pérdida = {loss(0, forward(network, M3)['p']):.3f}; "
      f"después: p̂ = {forward(new, M3)['p']:.3f}, pérdida = {loss(0, forward(new, M3)['p']):.3f}")

## Lección 7.4 — Aprender por pequeños lotes: minilotes y épocas

Red en su estado inicial (D7.1), doce mensajes del capítulo 5. Para seguir el cálculo a mano, solo se mueve la constante de salida $c$. Derivada de un lote = media de las derivadas de sus mensajes (ecuación 7.5).

In [ ]:
# Lección 7.4 — suma s, probabilidad y señal de error δ = p̂ − y de los doce mensajes, al inicio (c = 0)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def p_with_c(x, c_out):
    return forward(dict(network, c=c_out), x)['p']    # solo cambia la constante de salida


deltas = np.array([p_with_c(x, 0) - y_ for x, y_ in zip(X12, y12)])
print(pd.DataFrame({'message': twelve['message'], 'y': y12, 's': [forward(network, x)['z'] for x in X12],
                    'p̂': [p_with_c(x, 0) for x in X12], 'δ': deltas}).round(3).to_string(index=False))
print(f"derivada respecto a c — M7 solo: {deltas[6]:.3f}; lote M1 a M4: {deltas[:4].mean():.3f}; "
      f"los doce: suma {deltas.sum():.3f}, media {deltas.mean():.3f}")
print(f"lotes M5 a M8: {deltas[4:8].mean():.3f}; M9 a M12: {deltas[8:].mean():.3f}; "
      f"media de los tres lotes: {np.mean([deltas[:4].mean(), deltas[4:8].mean(), deltas[8:].mean()]):.3f}")


def loss_c(c_out):
    return np.mean([loss(y_, p_with_c(x, c_out)) for x, y_ in zip(X12, y12)])


# Una época: tres actualizaciones, cada una sobre su lote, cada una partiendo del valor que dejó la anterior
c_batch, path = 0.0, [0.0]
for k in range(3):
    batch = range(4 * k, 4 * k + 4)
    d = np.mean([p_with_c(X12[i], c_batch) - y12[i] for i in batch])
    c_batch = c_batch - 0.5 * d
    path.append(c_batch)
    print(f"actualización {k + 1} (M{4 * k + 1} a M{4 * k + 4}): derivada {d:.4f} → c = {c_batch:.3f}")
c_one_step = 0 - 0.5 * deltas.mean()
print(f"pérdida media de los doce: {loss_c(0):.3f} → {loss_c(c_batch):.3f} tras una época (3 actualizaciones); "
      f"un solo paso sobre los doce: c = {c_one_step:.3f}, pérdida {loss_c(c_one_step):.3f}")
plt.figure(figsize=(6, 2.6))
plt.plot(range(4), path, marker='o', color=FORWARD, label='una época, lotes de cuatro')
plt.plot([0, 1], [0, c_one_step], marker='s', color='black', label='un paso sobre los doce')
plt.plot([0, 1], [0, -0.5 * deltas[6]], marker='^', color=BACKWARD, linestyle='--', label='M7 solo')
plt.xlabel('actualización'); plt.ylabel('constante c'); plt.legend(); plt.grid(alpha=0.4); plt.show()

In [ ]:
# Preguntas cortas (lección 7.4)
updates_per_epoch = 12 // 3                      # n / m = 4
updates_ten_epochs = 10 * updates_per_epoch         # 40
c_batch_M6_M7 = 0 - 0.5 * deltas[5:7].mean()    # derivada ½ (0,500 − 0,330) ≈ 0,085, luego c ≈ −0,042

In [ ]:
# Comprobación de las preguntas cortas
if updates_per_epoch is None or updates_ten_epochs is None or c_batch_M6_M7 is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    print(f'1. {updates_per_epoch} por época, {updates_ten_epochs} para diez épocas →',
          'correcto' if (updates_per_epoch, updates_ten_epochs) == (4, 40) else 'revisar')
    print(f'2. c = {c_batch_M6_M7:.3f} →', 'correcto' if abs(c_batch_M6_M7 + 0.5 * deltas[5:7].mean()) < 1e-9 else 'revisar')

## Taller opcional — entrenar la red con los doce mensajes

1000 pasos de descenso de gradiente ($\eta = 0{,}5$), a partir de los pesos de la lección 7.1, promediando en cada paso las derivadas de los doce mensajes del capítulo 5. **Todas las medidas de abajo se refieren a estos doce mensajes, que sirven para el entrenamiento**: no dicen nada del comportamiento de la red con mensajes nuevos (lección 6.3).

In [ ]:
# Taller opcional: entrenamiento con los doce mensajes (medidas sobre los datos de entrenamiento)
X12 = list(zip(twelve['links'], twelve['suspicious_words']))
y12 = [1 if e == 'spam' else 0 for e in twelve['label']]


def loss_12(P):
    return np.mean([loss(y, forward(P, x)['p']) for x, y in zip(X12, y12)])


def gradient_12(P):
    gs = [backward(P, x, y) for x, y in zip(X12, y12)]
    return {k: np.mean([g[k] for g in gs]) for k in P}     # derivada de la pérdida media = media de las derivadas


P_trained, history = dict(network), [loss_12(network)]
for _ in range(1000):
    P_trained = update(P_trained, gradient_12(P_trained), 0.5)
    history.append(loss_12(P_trained))
print(f"pérdida media en los doce mensajes de entrenamiento: antes {history[0]:.3f}, después {history[-1]:.3f}")
for name, x in (('M7', M7), ('M3', M3)):
    print(f"{name}: p̂ antes {forward(network, x)['p']:.3f}, después {forward(P_trained, x)['p']:.3f}")
print('errores con el umbral 0,5 en los doce: antes', sum((forward(network, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)),
      ', después', sum((forward(P_trained, x)['p'] >= 0.5) != y for x, y in zip(X12, y12)))
print('pesos tras el entrenamiento:', {k: round(float(v), 2) for k, v in P_trained.items()})
plt.figure(figsize=(6, 3)); plt.plot(history, color=FORWARD)
plt.xlabel('paso'); plt.ylabel("pérdida media (doce mensajes de entrenamiento)"); plt.grid(alpha=0.4); plt.show()

## Ejercicio 7.1 — Leer la red de un colega

*Nociones: capas, pesos y sesgos (lección 7.1, ecuaciones 7.1 y 7.2).* Datos D7.E1: el diccionario `colleague`.

### a) a e) Solución comentada

**a)** Seis pesos, en la capa oculta ($w_{11}$, $w_{12}$, $w_{21}$, $w_{22}$) y en la salida ($v_1$, $v_2$), y tres sesgos ($b_1$, $b_2$, $c$): nueve parámetros, como la red de la lección 7.1.

**b)** $w_{21} = -1$ une los enlaces con la neurona N2; $w_{12} = 1$ une las palabras sospechosas con la neurona N1.

**c)** El sesgo −4 es el de N1. El sesgo de la salida vale $c = -2$.

**d)** Con $x_1 = x_2 = 0$, los productos por los pesos son nulos: $z_1 = b_1 = -4$ y $z_2 = b_2 = -1$. El sesgo es la suma ponderada de un mensaje «vacío».

**e)** Aquí, N2 calcula $-x_1 + 2x_2 - 1$: crece con las palabras sospechosas y disminuye con los enlaces, y su peso de salida $v_2 = 1$ es positivo: cuando su activación aumenta, la probabilidad de spam **sube**. En la red de la lección 7.1, N2 calculaba $x_1 - x_2$ y su peso de salida era negativo: es casi el efecto contrario.

In [ ]:
n_parameters = len(colleague)
empty = forward(colleague, (0, 0))

In [ ]:
# Comprobación de a) y d)
if n_parameters is None or empty is None:
    print('aún sin completar. Completa las dos líneas.')
else:
    print(f"{n_parameters} parámetros; mensaje vacío: z1 = {empty['z1']:g}, z2 = {empty['z2']:g} →",
          'correcto' if n_parameters == 9 and empty['z1'] == -4 and empty['z2'] == -1 else 'revisar')

## Ejercicio 7.2 — Calcular una salida a mano

*Nociones: propagación hacia delante (ecuaciones 7.1 y 7.2), sigmoide (ecuación 5.4), pérdida (ecuación 5.6), regla de decisión (ecuación 5.1).*

### a) a c) Solución comentada

**a)** M6: $z_1 = 1 \times 2 + 1 \times 2 - 4 = 0$, $a_1 = \sigma(0) = 0{,}5$; $z_2 = -1 \times 2 + 2 \times 2 - 1 = 1$, $a_2 = \sigma(1) \approx 0{,}7311$; $z = 2 \times 0{,}5 + 1 \times 0{,}7311 - 2 \approx -0{,}269$; $\hat p = \sigma(-0{,}269) \approx 0{,}433$.

**b)** $0{,}433 < 0{,}5$: M6 no se marca, lo cual es correcto, ya que es legítimo. Su pérdida vale $-\ln(1 - 0{,}433) \approx 0{,}568$.

**c)** M7: $z_1 = 2 + 3 - 4 = 1$, $a_1 \approx 0{,}7311$; $z_2 = -2 + 6 - 1 = 3$, $a_2 \approx 0{,}9526$; $z = 1{,}4621 + 0{,}9526 - 2 \approx 0{,}415$; $\hat p \approx 0{,}602$, pérdida $-\ln(0{,}602) \approx 0{,}507$. La red de la lección 7.1 le asigna a M7 una pérdida menor, 0,401: le da una probabilidad más cercana a 1.

In [ ]:
p_M6 = forward(colleague, (2, 2))['p']
loss_M6 = loss(0, p_M6)                          # M6 es legítimo: y = 0
loss_M7 = loss(1, forward(colleague, M7)['p'])      # M7 es spam: y = 1

In [ ]:
# Comprobación de a) a c)
if p_M6 is None or loss_M6 is None or loss_M7 is None:
    print('aún sin completar. Completa las tres líneas.')
else:
    print(f"M6: p̂ = {p_M6:.3f}, marcado: {p_M6 >= 0.5}, pérdida = {loss_M6:.3f} →",
          'correcto' if abs(p_M6 - 0.4332) < 5e-4 and abs(loss_M6 - 0.5677) < 5e-4 else 'revisar')
    print(f"M7: pérdida {loss_M7:.3f} (colega) frente a {loss(1, forward(network, M7)['p']):.3f} (lección 7.1) →",
          'correcto' if abs(loss_M7 - 0.5071) < 5e-4 else 'revisar')
    draw(colleague, (2, 2))

## Ejercicio 7.3 — Interpretar una activación

*Nociones: papel de una neurona oculta (lección 7.1).* Red de la lección 7.1, doce mensajes del capítulo 5.

### a) a d) Solución comentada

**a)** $a_2$ supera 0,8 para M2 (0,881) y M3 (0,982): dos mensajes legítimos, con enlaces y **ninguna** palabra sospechosa. $a_2$ vale exactamente 0,5 cuando $z_2 = x_1 - x_2 = 0$, es decir, cuando el mensaje tiene tantos enlaces como palabras sospechosas: M4, M6, M9.

**b)** N2 se enciende (activación por encima de 0,5) cuando un mensaje tiene más enlaces que palabras sospechosas, y más cuanto mayor es la diferencia. Su peso de salida $v_2 = -2$ hace entonces bajar la puntuación de salida y, por tanto, la probabilidad de spam. En estos seis mensajes, los dos en los que N2 se enciende más son legítimos; pero seis mensajes, con pesos elegidos a mano, no bastan para decir que N2 «detecta los boletines»: habría que comprobarlo con muchos mensajes nuevos.

**c)** M6: $z = 2 \times 0{,}5 - 2 \times 0{,}5 + 0 = 0$ exactamente, así que $\hat p = 0{,}5$ exactamente. Con la regla «mayor o igual», M6 se marca: una falsa alarma, justo en el umbral. Es la situación de M3 en el capítulo 5, cuya puntuación bruta valía exactamente 0.

**d)** Es N2: con $a_2 \approx 0{,}982$, resta unos 1,96 a la puntuación de salida, y la red ya solo le da 0,276 a M3. Una activación no es una probabilidad de spam: es un número entre 0 y 1 que sirve de ingrediente para la salida. Solo $\hat p$, en la salida, es una probabilidad estimada.

In [ ]:
twelve['a2'] = [forward(network, (l, m))['a2'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
lit_up = list(twelve.loc[twelve['a2'] > 0.8, 'message'])

In [ ]:
# Comprobación de a) y c)
if lit_up is None or twelve['a2'].isna().any():
    print('aún sin completar. Completa las dos líneas.')
else:
    twelve['a1'] = [forward(network, (l, m))['a1'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    twelve['p̂'] = [forward(network, (l, m))['p'] for l, m in zip(twelve['links'], twelve['suspicious_words'])]
    print(twelve.round(3).to_string(index=False))
    print('a2 > 0,8:', lit_up, '→', 'correcto' if lit_up == ['M2', 'M3'] else 'revisar')
    print(f"M6: z = {forward(network, (2, 2))['z']}, p̂ = {forward(network, (2, 2))['p']} → marcado con el umbral 0,5: {forward(network, (2, 2))['p'] >= 0.5}")

## Ejercicio 7.4 — Completar una derivada en Jupyter

*Nociones: regla de la cadena (ecuación 7.3), comprobación numérica (lección 7.2).* El peso $w_{22}$ une las palabras sospechosas con la neurona N2.

### a) y b) Solución comentada

**a)** ① $\hat p - y \approx -0{,}3303$; ② $v_2 = -2$; ③ $a_2(1 - a_2) = 0{,}2689 \times 0{,}7311 \approx 0{,}1966$; ④ $x_2 = 3$. Producto: $-0{,}3303 \times (-2) \times 0{,}1966 \times 3 \approx 0{,}390$.

**b)** El signo es positivo: aumentar $w_{22}$ hace **subir** la pérdida. Sigue los signos: un aumento de $w_{22}$ hace subir $z_2$ (porque $x_2 = 3 > 0$) y, por tanto, $a_2$; como $v_2 < 0$, $z$ baja; $\hat p$ baja, cuando M7 es un spam cuya probabilidad ya es demasiado baja. Dos factores negativos, ① y ②, dan un producto positivo.

In [ ]:
def derivative_w22(x, y):
    v = forward(network, x)
    return (v['p'] - y) * network['v2'] * v['a2'] * (1 - v['a2']) * x[1]   # ① × ② × ③ × ④

In [ ]:
# Comprobación de c): regla de la cadena frente a perturbación numérica
if derivative_w22(M7, 1) is None:
    print('aún sin completar. Completa la función.')
else:
    h = 0.0001
    num = (loss(1, forward(dict(network, w22=-1 + h), M7)['p']) - loss(1, forward(dict(network, w22=-1 - h), M7)['p'])) / (2 * h)
    print(f"regla de la cadena: {derivative_w22(M7, 1):.4f} →", 'correcto' if abs(derivative_w22(M7, 1) - 0.3896) < 5e-4 else 'revisar')
    print(f"derivada numérica: {num:.4f} → diferencia {abs(num - derivative_w22(M7, 1)):.1e} →",
          'correcto' if abs(num - derivative_w22(M7, 1)) < 1e-6 else 'revisar')

## Ejercicio 7.5 — Predecir y luego comprobar: el efecto de un cambio de peso

*Nociones: derivada y aproximación (lecciones 3.3 y 7.2).*

### a) Solución comentada

Solo con la derivada: pérdida $\approx 0{,}401 - 0{,}310 \times \Delta w$. Para $\Delta w = 0{,}1$: $0{,}401 - 0{,}031 = 0{,}370$. Para $\Delta w = 2$: $0{,}401 - 0{,}620 \approx -0{,}22$, una pérdida **negativa**, lo cual es imposible.

In [ ]:
def loss_after(dw):
    return loss(1, forward(dict(network, w11=0.5 + dw), M7)['p'])   # rehacemos toda la propagación hacia delante

In [ ]:
# Comprobación de b)
if loss_after(0.1) is None:
    print('aún sin completar. Completa la función.')
else:
    L0 = loss(1, forward(network, M7)['p'])
    for dw in (0.1, 2):
        print(f"dw = {dw}: predicha {L0 + d_w11 * dw:.3f}, exacta {loss_after(dw):.3f}")
    print('→', 'correcto' if abs(loss_after(0.1) - 0.3716) < 5e-4 and abs(loss_after(2) - 0.2126) < 5e-4 else 'revisar')

### b) y c) Solución comentada

Pérdidas exactas: 0,372 para $w_{11} = 0{,}6$, muy cerca del 0,370 predicho; 0,213 para $w_{11} = 2{,}5$, muy lejos del −0,22 predicho. La derivada describe la curva **cerca** del punto de partida: la tangente se pega a la curva para un paso pequeño y luego se aleja de ella (figura de la lección 7.2). Para un paso grande, la pérdida se aplana y no puede volverse negativa. Por eso la tasa de aprendizaje se mantiene pequeña: cada paso debe quedarse en la zona donde la derivada dice la verdad, aunque haya que dar muchos.

## Ejercicio 7.6 — Reto: actualizar varios pesos con dos mensajes

*Nociones: retropropagación (ecuación 7.4), pérdida media (ecuación 5.7), descenso de gradiente (ecuación 3.7).* Mensajes M7 (spam) y M3 (legítimo).

### a) a c) Solución comentada

**a)** Pérdidas de partida: 0,401 para M7 y $-\ln(1 - 0{,}2761) \approx 0{,}323$ para M3; media $\approx 0{,}362$.

**b)** $\delta_{M7} = 0{,}6697 - 1 \approx -0{,}330$ y $\delta_{M3} = 0{,}2761 - 0 = 0{,}2761$. Derivadas de la pérdida media: $\partial L/\partial v_1 = (-0{,}330 \times 0{,}6225 + 0{,}2761 \times 0{,}5)/2 \approx -0{,}034$; $\partial L/\partial v_2 = (-0{,}330 \times 0{,}2689 + 0{,}2761 \times 0{,}9820)/2 \approx 0{,}091$; $\partial L/\partial c = (-0{,}330 + 0{,}2761)/2 \approx -0{,}027$.

**c)** Con $\eta = 1$: $v_1 \approx 2{,}034$, $v_2 \approx -2{,}091$, $c \approx 0{,}027$. Las activaciones ocultas no cambian. Nuevas pérdidas: alrededor de 0,393 para M7 y 0,311 para M3, media 0,352. La pérdida media baja, y **cada una** de las dos también: a diferencia de la actualización solo con M7 (lección 7.3), M3 no se sacrifica, porque su derivada entra en la media.

In [ ]:
def mean_gradient(P):
    g7, g3 = backward(P, M7, 1), backward(P, M3, 0)
    return {k: (g7[k] + g3[k]) / 2 for k in P}    # derivada de una media = media de las derivadas

In [ ]:
# Comprobación de c) y luego d): los nueve parámetros, un paso y luego 100 pasos
def mean_loss(P):
    return (loss(1, forward(P, M7)['p']) + loss(0, forward(P, M3)['p'])) / 2

if mean_gradient(network) is None:
    print('aún sin completar. Completa la función.')
else:
    g = mean_gradient(network)
    output_only = dict(network, v1=network['v1'] - g['v1'], v2=network['v2'] - g['v2'], c=network['c'] - g['c'])
    print(f"inicio: pérdida media {mean_loss(network):.3f}")
    print(f"c) solo v1, v2, c: M7 {loss(1, forward(output_only, M7)['p']):.3f}, "
          f"M3 {loss(0, forward(output_only, M3)['p']):.3f}, media {mean_loss(output_only):.3f} →",
          'correcto' if abs(mean_loss(output_only) - 0.3519) < 5e-4 else 'revisar')
    nine = update(network, g, 1)
    print(f"d) los nueve parámetros, un paso: media {mean_loss(nine):.3f} →",
          'correcto' if abs(mean_loss(nine) - 0.2584) < 5e-4 else 'revisar')
    P, history = dict(network), []
    for _ in range(100):
        P = update(P, mean_gradient(P), 1)
        history.append(mean_loss(P))
    print(f"   100 pasos: p̂(M7) = {forward(P, M7)['p']:.3f}, p̂(M3) = {forward(P, M3)['p']:.3f}, pérdida media {history[-1]:.3f}")
    plt.figure(figsize=(6, 3)); plt.plot(range(1, 101), history, color=FORWARD)
    plt.xlabel('paso'); plt.ylabel('pérdida media (M7 y M3)'); plt.grid(alpha=0.4); plt.show()

### d) Solución comentada

Un paso sobre los nueve parámetros hace caer la pérdida media de 0,362 a 0,258: mucho más que la capa de salida sola (0,352), porque las neuronas ocultas también se adaptan. Tras 100 pasos, la red da 0,982 a M7 y 0,018 a M3: pérdida media 0,018. Este éxito no prueba nada para lo que viene: con nueve parámetros para dos mensajes, la red puede aprenderlos de memoria, como el filtro B de la lección 6.3. Solo mensajes que no ha visto dirían lo que vale.